#Creamos schema y volumen para almacenar nuestro database


In [0]:
%sql
-- 1. Creamos el esquema específico para el proyecto
CREATE SCHEMA IF NOT EXISTS workspace.tp_dnrpa;

-- 2. Creamos el volumen dentro de ese nuevo esquema
CREATE VOLUME IF NOT EXISTS workspace.tp_dnrpa.dnrpa_landing;

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .load("/Volumes/workspace/tp_dnrpa/dnrpa_landing/Origenes/**/*.csv")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.tp_dnrpa.dnrpa_landing")

#1- Exploración Inicial

## 1.1 Verificamos datos disponibles


In [0]:
%sql 
SELECT count(*) as total_registros from workspace.tp_dnrpa.dnrpa_landing

##1.2 Verificamos estructura y tipo de dato


In [0]:
%sql

DESCRIBE workspace.tp_dnrpa.dnrpa_landing

##1.3 Verificamos muestreo de datos "landing"

In [0]:
%sql

SELECT * 
FROM workspace.tp_dnrpa.dnrpa_landing
LIMIT 100;


#2- Análisis de Valores Nullos

##2.1 Nulos por columna

In [0]:
%sql
WITH total AS (
    SELECT COUNT(*) as n 
    FROM workspace.tp_dnrpa.dnrpa_landing
),
nulos AS (
    SELECT
        COUNT(*) - COUNT(tramite_tipo) as nulos_tramite_tipo,
        COUNT(*) - COUNT(tramite_fecha) as nulos_tramite_fecha,
        COUNT(*) - COUNT(fecha_inscripcion_inicial) as nulos_fecha_inscripcion_inicial,
        COUNT(*) - COUNT(registro_seccional_codigo) as nulos_registro_seccional_codigo,
        COUNT(*) - COUNT(registro_seccional_descripcion) as nulos_registro_seccional_descripcion,
        COUNT(*) - COUNT(registro_seccional_provincia) as nulos_registro_seccional_provincia,
        COUNT(*) - COUNT(automotor_origen) as nulos_automotor_origen,
        COUNT(*) - COUNT(automotor_anio_modelo) as nulos_automotor_anio_modelo,
        COUNT(*) - COUNT(automotor_tipo_codigo) as nulos_automotor_tipo_codigo,
        COUNT(*) - COUNT(automotor_tipo_descripcion) as nulos_automotor_tipo_descripcion,
        COUNT(*) - COUNT(automotor_marca_codigo) as nulos_automotor_marca_codigo,
        COUNT(*) - COUNT(automotor_marca_descripcion) as nulos_automotor_marca_descripcion,
        COUNT(*) - COUNT(automotor_modelo_codigo) as nulos_automotor_modelo_codigo,
        COUNT(*) - COUNT(automotor_modelo_descripcion) as nulos_automotor_modelo_descripcion,
        COUNT(*) - COUNT(automotor_uso_codigo) as nulos_automotor_uso_codigo,
        COUNT(*) - COUNT(automotor_uso_descripcion) as nulos_automotor_uso_descripcion,
        COUNT(*) - COUNT(titular_tipo_persona) as nulos_titular_tipo_persona,
        COUNT(*) - COUNT(titular_domicilio_localidad) as nulos_titular_domicilio_localidad,
        COUNT(*) - COUNT(titular_domicilio_provincia) as nulos_titular_domicilio_provincia,
        COUNT(*) - COUNT(titular_genero) as nulos_titular_genero,
        COUNT(*) - COUNT(titular_anio_nacimiento) as nulos_titular_anio_nacimiento,
        COUNT(*) - COUNT(titular_pais_nacimiento) as nulos_titular_pais_nacimiento,
        COUNT(*) - COUNT(titular_porcentaje_titularidad) as nulos_titular_porcentaje_titularidad,
        COUNT(*) - COUNT(titular_domicilio_provincia_id) as nulos_titular_domicilio_provincia_id,
        COUNT(*) - COUNT(titular_pais_nacimiento_id) as nulos_titular_pais_nacimiento_id
    FROM workspace.tp_dnrpa.dnrpa_landing
)
SELECT 
    t.n as total_registros,
    n.*
FROM total t, nulos n

##2.2 Porcentaje de nulos

In [0]:
%sql
WITH total AS (
    SELECT COUNT(*) as total FROM workspace.tp_dnrpa.dnrpa_landing
)
SELECT 
    ROUND((t.total - COUNT(tramite_fecha)) * 100.0 / t.total, 2) as pct_nulos_tramite_fecha,
    ROUND((t.total - COUNT(fecha_inscripcion_inicial)) * 100.0 / t.total, 2) as pct_nulos_fecha_inscripcion_inicial,
    ROUND((t.total - COUNT(registro_seccional_codigo)) * 100.0 / t.total, 2) as pct_nulos_registro_seccional_codigo,
    ROUND((t.total - COUNT(registro_seccional_descripcion)) * 100.0 / t.total, 2) as pct_nulos_registro_seccional_descripcion,
    ROUND((t.total - COUNT(registro_seccional_provincia)) * 100.0 / t.total, 2) as pct_nulos_registro_seccional_provincia,
    ROUND((t.total - COUNT(automotor_origen)) * 100.0 / t.total, 2) as pct_nulos_automotor_origen,
    ROUND((t.total - COUNT(automotor_anio_modelo)) * 100.0 / t.total, 2) as pct_nulos_automotor_anio_modelo,
    ROUND((t.total - COUNT(automotor_tipo_codigo)) * 100.0 / t.total, 2) as pct_nulos_automotor_tipo_codigo,
    ROUND((t.total - COUNT(automotor_tipo_descripcion)) * 100.0 / t.total, 2) as pct_nulos_automotor_tipo_descripcion,
    ROUND((t.total - COUNT(automotor_marca_codigo)) * 100.0 / t.total, 2) as pct_nulos_automotor_marca_codigo,
    ROUND((t.total - COUNT(automotor_marca_descripcion)) * 100.0 / t.total, 2) as pct_nulos_automotor_marca_descripcion,
    ROUND((t.total - COUNT(automotor_modelo_codigo)) * 100.0 / t.total, 2) as pct_nulos_automotor_modelo_codigo,
    ROUND((t.total - COUNT(automotor_modelo_descripcion)) * 100.0 / t.total, 2) as pct_nulos_automotor_modelo_descripcion,
    ROUND((t.total - COUNT(automotor_uso_codigo)) * 100.0 / t.total, 2) as pct_nulos_automotor_uso_codigo,
    ROUND((t.total - COUNT(automotor_uso_descripcion)) * 100.0 / t.total, 2) as pct_nulos_automotor_uso_descripcion,
    ROUND((t.total - COUNT(titular_tipo_persona)) * 100.0 / t.total, 2) as pct_nulos_titular_tipo_persona,
    ROUND((t.total - COUNT(titular_domicilio_localidad)) * 100.0 / t.total, 2) as pct_nulos_titular_domicilio_localidad,
    ROUND((t.total - COUNT(titular_domicilio_provincia)) * 100.0 / t.total, 2) as pct_nulos_titular_domicilio_provincia,
    ROUND((t.total - COUNT(titular_genero)) * 100.0 / t.total, 2) as pct_nulos_titular_genero,
    ROUND((t.total - COUNT(titular_anio_nacimiento)) * 100.0 / t.total, 2) as pct_nulos_titular_anio_nacimiento,
    ROUND((t.total - COUNT(titular_pais_nacimiento)) * 100.0 / t.total, 2) as pct_nulos_titular_pais_nacimiento,
    ROUND((t.total - COUNT(titular_porcentaje_titularidad)) * 100.0 / t.total, 2) as pct_nulos_titular_porcentaje_titularidad,
    ROUND((t.total - COUNT(titular_domicilio_provincia_id)) * 100.0 / t.total, 2) as pct_nulos_titular_domicilio_provincia_id,
    ROUND((t.total - COUNT(titular_pais_nacimiento_id)) * 100.0 / t.total, 2) as pct_nulos_titular_pais_nacimiento_id
FROM workspace.tp_dnrpa.dnrpa_landing
CROSS JOIN total t
GROUP BY t.total;

##2.3 Verificamos columnas críticas

In [0]:
%sql
WITH total AS (
    SELECT COUNT(*) as total FROM workspace.tp_dnrpa.dnrpa_landing
),
pct_nulos AS (
    SELECT 
        ROUND((t.total - COUNT(tramite_fecha)) * 100.0 / t.total, 2) as tramite_fecha,
        ROUND((t.total - COUNT(fecha_inscripcion_inicial)) * 100.0 / t.total, 2) as fecha_inscripcion_inicial,
        ROUND((t.total - COUNT(registro_seccional_codigo)) * 100.0 / t.total, 2) as registro_seccional_codigo,
        ROUND((t.total - COUNT(registro_seccional_descripcion)) * 100.0 / t.total, 2) as registro_seccional_descripcion,
        ROUND((t.total - COUNT(registro_seccional_provincia)) * 100.0 / t.total, 2) as registro_seccional_provincia,
        ROUND((t.total - COUNT(automotor_origen)) * 100.0 / t.total, 2) as automotor_origen,
        ROUND((t.total - COUNT(automotor_anio_modelo)) * 100.0 / t.total, 2) as automotor_anio_modelo,
        ROUND((t.total - COUNT(automotor_tipo_codigo)) * 100.0 / t.total, 2) as automotor_tipo_codigo,
        ROUND((t.total - COUNT(automotor_tipo_descripcion)) * 100.0 / t.total, 2) as automotor_tipo_descripcion,
        ROUND((t.total - COUNT(automotor_marca_codigo)) * 100.0 / t.total, 2) as automotor_marca_codigo,
        ROUND((t.total - COUNT(automotor_marca_descripcion)) * 100.0 / t.total, 2) as automotor_marca_descripcion,
        ROUND((t.total - COUNT(automotor_modelo_codigo)) * 100.0 / t.total, 2) as automotor_modelo_codigo,
        ROUND((t.total - COUNT(automotor_modelo_descripcion)) * 100.0 / t.total, 2) as automotor_modelo_descripcion,
        ROUND((t.total - COUNT(automotor_uso_codigo)) * 100.0 / t.total, 2) as automotor_uso_codigo,
        ROUND((t.total - COUNT(automotor_uso_descripcion)) * 100.0 / t.total, 2) as automotor_uso_descripcion,
        ROUND((t.total - COUNT(titular_tipo_persona)) * 100.0 / t.total, 2) as titular_tipo_persona,
        ROUND((t.total - COUNT(titular_domicilio_localidad)) * 100.0 / t.total, 2) as titular_domicilio_localidad,
        ROUND((t.total - COUNT(titular_domicilio_provincia)) * 100.0 / t.total, 2) as titular_domicilio_provincia,
        ROUND((t.total - COUNT(titular_genero)) * 100.0 / t.total, 2) as titular_genero,
        ROUND((t.total - COUNT(titular_anio_nacimiento)) * 100.0 / t.total, 2) as titular_anio_nacimiento,
        ROUND((t.total - COUNT(titular_pais_nacimiento)) * 100.0 / t.total, 2) as titular_pais_nacimiento,
        ROUND((t.total - COUNT(titular_porcentaje_titularidad)) * 100.0 / t.total, 2) as titular_porcentaje_titularidad,
        ROUND((t.total - COUNT(titular_domicilio_provincia_id)) * 100.0 / t.total, 2) as titular_domicilio_provincia_id,
        ROUND((t.total - COUNT(titular_pais_nacimiento_id)) * 100.0 / t.total, 2) as titular_pais_nacimiento_id
    FROM workspace.tp_dnrpa.dnrpa_landing
    CROSS JOIN total t
    GROUP BY t.total
)
SELECT columna, porcentaje_nulos
FROM (
    SELECT 'tramite_fecha' as columna, tramite_fecha as porcentaje_nulos FROM pct_nulos UNION ALL
    SELECT 'fecha_inscripcion_inicial', fecha_inscripcion_inicial FROM pct_nulos UNION ALL
    SELECT 'registro_seccional_codigo', registro_seccional_codigo FROM pct_nulos UNION ALL
    SELECT 'registro_seccional_descripcion', registro_seccional_descripcion FROM pct_nulos UNION ALL
    SELECT 'registro_seccional_provincia', registro_seccional_provincia FROM pct_nulos UNION ALL
    SELECT 'automotor_origen', automotor_origen FROM pct_nulos UNION ALL
    SELECT 'automotor_anio_modelo', automotor_anio_modelo FROM pct_nulos UNION ALL
    SELECT 'automotor_tipo_codigo', automotor_tipo_codigo FROM pct_nulos UNION ALL
    SELECT 'automotor_tipo_descripcion', automotor_tipo_descripcion FROM pct_nulos UNION ALL
    SELECT 'automotor_marca_codigo', automotor_marca_codigo FROM pct_nulos UNION ALL
    SELECT 'automotor_marca_descripcion', automotor_marca_descripcion FROM pct_nulos UNION ALL
    SELECT 'automotor_modelo_codigo', automotor_modelo_codigo FROM pct_nulos UNION ALL
    SELECT 'automotor_modelo_descripcion', automotor_modelo_descripcion FROM pct_nulos UNION ALL
    SELECT 'automotor_uso_codigo', automotor_uso_codigo FROM pct_nulos UNION ALL
    SELECT 'automotor_uso_descripcion', automotor_uso_descripcion FROM pct_nulos UNION ALL
    SELECT 'titular_tipo_persona', titular_tipo_persona FROM pct_nulos UNION ALL
    SELECT 'titular_domicilio_localidad', titular_domicilio_localidad FROM pct_nulos UNION ALL
    SELECT 'titular_domicilio_provincia', titular_domicilio_provincia FROM pct_nulos UNION ALL
    SELECT 'titular_genero', titular_genero FROM pct_nulos UNION ALL
    SELECT 'titular_anio_nacimiento', titular_anio_nacimiento FROM pct_nulos UNION ALL
    SELECT 'titular_pais_nacimiento', titular_pais_nacimiento FROM pct_nulos UNION ALL
    SELECT 'titular_porcentaje_titularidad', titular_porcentaje_titularidad FROM pct_nulos UNION ALL
    SELECT 'titular_domicilio_provincia_id', titular_domicilio_provincia_id FROM pct_nulos UNION ALL
    SELECT 'titular_pais_nacimiento_id', titular_pais_nacimiento_id FROM pct_nulos
) unpivoted
WHERE porcentaje_nulos > 5
ORDER BY porcentaje_nulos DESC

#3- Cardinalidad y Distribución

##3.1 Distribución de tipo de tramite

In [0]:
%sql
SELECT 
  tramite_tipo,
  COUNT(*) as cantidad_registros,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM workspace.tp_dnrpa.dnrpa_landing
WHERE tramite_tipo IS NOT NULL
GROUP BY tramite_tipo
ORDER BY cantidad_registros DESC

##3.2 Distribución de marca de automotor

In [0]:
%sql
SELECT 
  automotor_marca_descripcion,
  COUNT(*) as cantidad_registros,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM workspace.tp_dnrpa.dnrpa_landing
WHERE automotor_marca_descripcion IS NOT NULL
GROUP BY automotor_marca_descripcion
ORDER BY cantidad_registros DESC

##3.3 Distribución por modelo 

In [0]:
%sql
SELECT 
  automotor_modelo_descripcion,
  COUNT(*) as cantidad_registros,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM workspace.tp_dnrpa.dnrpa_landing
WHERE automotor_modelo_descripcion IS NOT NULL
GROUP BY automotor_modelo_descripcion
ORDER BY cantidad_registros DESC

##3.4 Top Provincias

In [0]:
%sql
SELECT 
  titular_domicilio_provincia,
  COUNT(*) as cantidad_registros,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM workspace.tp_dnrpa.dnrpa_landing
WHERE titular_domicilio_provincia IS NOT NULL
GROUP BY titular_domicilio_provincia
ORDER BY cantidad_registros DESC
LIMIT 30;

##3.5 Distribución por Tipo descripción

In [0]:
%sql
SELECT 
 automotor_tipo_descripcion,
  COUNT(*) as cantidad_registros,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM workspace.tp_dnrpa.dnrpa_landing
WHERE automotor_tipo_descripcion IS NOT NULL
GROUP BY automotor_tipo_descripcion
ORDER BY cantidad_registros DESC

#4- Descripción de estadísticas y variables numéricas

##4.1 Estadística de modelo por año

In [0]:
%sql
SELECT COUNT (*) as TOTAL_REGISTROS,
  automotor_modelo_descripcion,
  ROUND (AVG(CAST(automotor_anio_modelo AS STRING)),2) AS ANIO_PROMEDIO,
  ROUND (MAX(CAST(automotor_anio_modelo AS STRING)),2) AS ANIO_MAXIMO,
  ROUND (MIN(CAST(automotor_anio_modelo AS STRING)),2) AS ANIO_MINIMO,
  ROUND (PERCENTILE(CAST(automotor_anio_modelo AS STRING), 0.25),2) AS PERCENTIL_25,
  ROUND (PERCENTILE(CAST(automotor_anio_modelo AS STRING), 0.75),2) AS PERCENTIL_75,
  ROUND (PERCENTILE(CAST(automotor_anio_modelo AS STRING), 0.5),2) AS MEDIANA
FROM workspace.tp_dnrpa.dnrpa_landing
WHERE automotor_modelo_descripcion IS NOT NULL AND CAST(automotor_anio_modelo AS STRING) IS NOT NULL AND CAST(automotor_anio_modelo AS STRING) > '0'
GROUP BY automotor_modelo_descripcion
ORDER BY TOTAL_REGISTROS DESC;

##4.2 Estadística de tipo de descripción

In [0]:
%sql
SELECT 
  automotor_marca_descripcion,
  automotor_tipo_descripcion,
  COUNT(*) as TOTAL_REGISTROS,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY automotor_marca_descripcion), 2) as PORCENTAJE_DENTRO_MARCA
FROM workspace.tp_dnrpa.dnrpa_landing
WHERE automotor_marca_descripcion IS NOT NULL 
  AND automotor_tipo_descripcion IS NOT NULL
GROUP BY automotor_marca_descripcion, automotor_tipo_descripcion
ORDER BY  TOTAL_REGISTROS DESC;

##4.3 Análisis de antiguedad y tipo de tramite

In [0]:
%sql

select COUNT(*) as total_registros,
tramite_tipo,
automotor_anio_modelo,
ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
from workspace.tp_dnrpa.dnrpa_landing
where automotor_anio_modelo is not null
group by automotor_anio_modelo, tramite_tipo
order by total_registros desc 
limit 20;

##4.4 Estadística de antiguedad sin placeholders

In [0]:
%sql 

SELECT
  COUNT(*) AS cantidad,
  MIN(automotor_anio_modelo) AS minimo,
  MAX(automotor_anio_modelo) AS maximo,
  ROUND(AVG(automotor_anio_modelo), 2) AS promedio,
  ROUND(PERCENTILE(automotor_anio_modelo, 0.5), 2) AS mediana
FROM workspace.tp_dnrpa.dnrpa_landing
WHERE automotor_anio_modelo != 999
  AND automotor_anio_modelo IS NOT NULL
  AND automotor_anio_modelo >= 0;

#5- Detección de problemas de calidad

##5.1 Reporte de calidad

In [0]:
%sql
WITH total AS (
  SELECT COUNT(*) AS total_registros
  FROM workspace.tp_dnrpa.dnrpa_landing
),
problemas AS (
  SELECT
    COUNT(*) FILTER (WHERE automotor_anio_modelo IS NULL OR automotor_anio_modelo <= 0) AS anio_invalido,
    COUNT(*) FILTER (WHERE automotor_anio_modelo = 999) AS anio_placeholder,
    COUNT(*) FILTER (WHERE tramite_fecha IS NULL) AS fecha_vacia,
    COUNT(*) FILTER (WHERE automotor_marca_descripcion IS NULL OR automotor_marca_descripcion = '') AS marca_vacia,
    COUNT(*) FILTER (WHERE automotor_modelo_descripcion IS NULL OR automotor_modelo_descripcion = '') AS modelo_vacio,
    COUNT(*) FILTER (WHERE titular_domicilio_provincia IS NULL OR titular_domicilio_provincia = '') AS provincia_vacia
  FROM workspace.tp_dnrpa.dnrpa_landing
)
SELECT
  t.total_registros,
  p.anio_invalido,
  ROUND(p.anio_invalido * 100.0 / t.total_registros, 2) AS porcentaje_anio_invalido,
  p.anio_placeholder,
  ROUND(p.anio_placeholder * 100.0 / t.total_registros, 2) AS porcentaje_anio_placeholder,
  p.fecha_vacia,
  ROUND(p.fecha_vacia * 100.0 / t.total_registros, 2) AS porcentaje_fecha_vacia,
  p.marca_vacia,
  ROUND(p.marca_vacia * 100.0 / t.total_registros, 2) AS porcentaje_marca_vacia,
  p.modelo_vacio,
  ROUND(p.modelo_vacio * 100.0 / t.total_registros, 2) AS porcentaje_modelo_vacio,
  p.provincia_vacia,
  ROUND(p.provincia_vacia * 100.0 / t.total_registros, 2) AS porcentaje_provincia_vacia
FROM total t
CROSS JOIN problemas p

##5.2 Detectar duplicados

In [0]:
%sql
WITH marca_modelo_duplicados AS (
  SELECT 
    automotor_marca_descripcion,
    automotor_modelo_descripcion,
    COUNT(*) AS cantidad
  FROM workspace.tp_dnrpa.dnrpa_landing
  GROUP BY automotor_marca_descripcion, automotor_modelo_descripcion
  HAVING COUNT(*) > 1
)
SELECT
  COUNT(*) AS cantidad_grupos_duplicados,
  SUM(cantidad) AS total_registros_duplicados,
  SUM(cantidad) - COUNT(*) AS registros_extra
FROM marca_modelo_duplicados


##5.3 Ver ejemplos de duplicados

In [0]:
%sql
WITH duplicados AS (
    SELECT 
        automotor_marca_descripcion,
        automotor_modelo_descripcion,
        COUNT(*) as veces
    FROM workspace.tp_dnrpa.dnrpa_landing
    GROUP BY automotor_marca_descripcion, automotor_modelo_descripcion
    HAVING COUNT(*) > 1
)
SELECT *
FROM duplicados
ORDER BY veces DESC
LIMIT 10;

#6- Análisis Avanzado y documentación

##6.1 Ranking de provincias basado en el volumen total de transferencias registradas.

In [0]:
%sql
SELECT 
    registro_seccional_provincia AS provincia,
    COUNT(1) AS cantidad_transferencias,
    CAST(AVG(automotor_anio_modelo) AS INT) AS anio_modelo_promedio,
    ROW_NUMBER() OVER (ORDER BY COUNT(1) DESC) AS ranking_volumen
FROM workspace.tp_dnrpa.dnrpa_landing
WHERE registro_seccional_provincia IS NOT NULL 
  AND automotor_anio_modelo IS NOT NULL
GROUP BY registro_seccional_provincia
ORDER BY ranking_volumen ASC;

##6.2 Promedio de año por provincia

In [0]:
%sql
WITH PromediosProvinciales AS (
    SELECT 
        registro_seccional_provincia AS provincia,
        AVG(automotor_anio_modelo) AS promedio_provincial
    FROM workspace.tp_dnrpa.dnrpa_landing
    WHERE registro_seccional_provincia IS NOT NULL 
      AND automotor_anio_modelo IS NOT NULL
    GROUP BY registro_seccional_provincia
)
SELECT 
    provincia,
    CAST(promedio_provincial AS INT) AS anio_promedio_provincia,    
    CAST(AVG(promedio_provincial) OVER () AS INT) AS anio_promedio_nacional,  
    CAST(promedio_provincial - AVG(promedio_provincial) OVER () AS INT) AS diferencia_anios,
    ROUND(((promedio_provincial / AVG(promedio_provincial) OVER ()) - 1) * 100, 2) AS porcentaje_diferencia
FROM PromediosProvinciales
ORDER BY porcentaje_diferencia DESC;

##6.3 Análisis temporal

In [0]:
%sql
SELECT 
    DATE_TRUNC('month', CAST(tramite_fecha AS DATE)) AS mes_tramite,
    COUNT(1) AS cantidad_transferencias,
    CAST(AVG(automotor_anio_modelo) AS INT) AS anio_modelo_promedio
FROM workspace.tp_dnrpa.dnrpa_landing
WHERE tramite_fecha IS NOT NULL
GROUP BY DATE_TRUNC('month', CAST(tramite_fecha AS DATE))
ORDER BY mes_tramite ASC;

##6.4 Resumen de calidad del dato

In [0]:
%sql
WITH MetricasCalidad AS (
    
    SELECT 
        COUNT(1) AS total_registros,
        SUM(CASE WHEN registro_seccional_provincia IS NULL 
                   OR automotor_anio_modelo IS NULL 
                   OR tramite_fecha IS NULL 
                THEN 1 ELSE 0 END) AS registros_invalidos,             
                SUM(CASE WHEN registro_seccional_provincia IS NOT NULL 
                  AND automotor_anio_modelo IS NOT NULL 
                  AND tramite_fecha IS NOT NULL 
                     THEN 1 ELSE 0 END) AS registros_validos             
    FROM workspace.tp_dnrpa.dnrpa_landing
)
SELECT 
    total_registros,
    registros_invalidos,    
    ROUND((registros_validos / total_registros) * 100, 2) AS porcentaje_validos,
    ROUND((registros_invalidos / total_registros) * 100, 2) AS porcentaje_invalidos,
    'REGLAS SILVER: 1) Descartar filas con provincia nula. 2) Castear tramite_fecha a tipo DATE. 3) Filtrar años de modelo fuera de rango logico.' AS recomendaciones_silver
FROM MetricasCalidad;